Function for automatic segmentation copied from https://github.com/computational-cell-analytics/micro-sam/blob/master/notebooks/automatic_segmentation.ipynb

In [3]:
from glob import glob
from typing import Optional, Union, Tuple

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from tqdm import tqdm

from skimage.measure import label as connected_components
from skimage.measure import regionprops_table
from skimage.color import label2rgb

from tifffile import imread,imwrite

import torch

from micro_sam.evaluation.model_comparison import _enhance_image
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation

cuda GPU being used


In [4]:
def run_automatic_instance_segmentation(
    image: np.ndarray,
    ndim: int,
    checkpoint_path: Optional[Union[os.PathLike, str]] = None,
    model_type: str = "vit_b_lm",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
):
    """Automatic Instance Segmentation (AIS) by training an additional instance decoder in SAM.

    NOTE: AIS is supported only for `µsam` models.

    Args:
        image: The input image.
        ndim: The number of dimensions for the input data.
        checkpoint_path: The path to stored checkpoints.
        model_type: The choice of the `µsam` model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching the segmented tiles.

    Returns:
        The instance segmentation.
    """
    # Step 1: Get the 'predictor' and 'segmenter' to perform automatic instance segmentation.
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,  # choice of the Segment Anything model
        checkpoint=checkpoint_path,  # overwrite to pass your own finetuned model.
        device=device,  # the device to run the model inference.
        segmentation_mode="ais",  # set the automatic segmentation mode to AIS.
        is_tiled=(tile_shape is not None),  # whether to run automatic segmentation with tiling.
    )

    # Step 2: Get the instance segmentation for the given image.
    prediction = automatic_instance_segmentation(
        predictor=predictor,  # the predictor for the Segment Anything model.
        segmenter=segmenter,  # the segmenter class responsible for generating predictions.
        input_path=image,  # the filepath to image or the input array for automatic segmentation.
        ndim=ndim,  # the number of input dimensions.
        tile_shape=tile_shape,  # the tile shape for tiling-based prediction.
        halo=halo,  # the overlap shape for tiling-based prediction.
    )

    return prediction

In [5]:
def BorderRemoval(mask:np.array):

    def BorderElements(array:np.array, width:int): 
    
        n = array.shape[0]
        r = np.minimum(np.arange(n)[::-1], np.arange(n))
    
        a =  array[np.minimum(r[:,None],r)<width]

        return a[a.nonzero()]

    
    borderIDs = BorderElements(mask,2)

    if len(borderIDs) == 0:
        return mask

    else:
        
        CopyArray = np.copy(mask)

        for ID in borderIDs:

            Negative_mask = (mask != ID)
        
            CopyArray *= Negative_mask
    
        return CopyArray

### Run the segmentation loop

In [ ]:
imgs = sorted(glob('../../groups/evocell/Octavio/Omnipose/Benchmarking/Sampling_test_FMSeg/Deconvolved/Images/*.tif'))
print(len(imgs))

exdir = '../../groups/evocell/Octavio/Omnipose/Benchmarking/Sampling_test_FMSeg/Deconvolved/microSAM/'

fm_model_path = '../../groups/evocell/Octavio/sam/models/checkpoints/microSAM_Vegetative_FM4/best.pt'


for i in tqdm(range(len(imgs))):

    img = imread(imgs[i])

    imname = imgs[i].split('/')[-1]
    
    prediction_fm = run_automatic_instance_segmentation(img, ndim=2,
                                                    model_type="vit_b_lm",
                                                    checkpoint_path=fm_model_path,
                                                    tile_shape=(256,256),
                                                    halo=(64,64))

    imwrite(f'{exdir}{imname}',np.uint16(prediction_fm))


56


Compute Image Embeddings 2D tiled: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:22<00:00,  2.80it/s]

Initialize tiled instance segmentation with decoder:   0%|                                                                                                                                            | 0/64 [00:00<?, ?it/s]
Initialize tiled instance segmentation with decoder:   2%|██                                                                                                                                  | 1/64 [00:02<02:20,  2.22s/it]
Initialize tiled instance segmentation with decoder:   3%|████▏                                                                                                                               | 2/64 [00:02<01:01,  1.01it/s]
Initialize tiled instance segmentation with decoder:   5%|██████▏                                              